# Lab 7 — Transfer Learning Quantique

## Objectifs
- Implémenter une architecture hybride : CNN classique (gelé) + tête quantique
- Comparer avec une tête classique fully-connected
- Analyser la convergence et l'accuracy

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, Subset
import pennylane as qml
from pennylane import qnode
import numpy as np
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import time

In [ ]:
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

num_classes = 4
batch_size = 32
n_epochs = 50
lr = 0.01

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

## Partie A — Extracteur de features classique (ResNet18 gelé)

In [ ]:
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in resnet.parameters():
    param.requires_grad = False

feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
feature_extractor.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = CIFAR10(root="./data", train=False, download=True, transform=transform)

selected_classes = [0, 1, 2, 3]
train_idx = [i for i, (_, label) in enumerate(train_dataset) if label in selected_classes]
test_idx = [i for i, (_, label) in enumerate(test_dataset) if label in selected_classes]

train_subset = Subset(train_dataset, train_idx[:2000])
test_subset = Subset(test_dataset, test_idx[:400])

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

def extract_features(dataloader, model):
    model.eval()
    all_features = []
    all_labels = []
    with torch.no_grad():
        for images, labels in dataloader:
            features = model(images)
            features = features.squeeze()
            all_features.append(features)
            all_labels.append(labels)
    return torch.cat(all_features), torch.cat(all_labels)

X_train, y_train = extract_features(train_loader, feature_extractor)
X_test, y_test = extract_features(test_loader, feature_extractor)

pca = nn.Linear(512, n_qubits)
nn.init.xavier_uniform_(pca.weight)

X_train_reduced = torch.relu(pca(X_train)).detach()
X_test_reduced = torch.relu(pca(X_test)).detach()

X_train_norm = X_train_reduced / (X_train_reduced.max(dim=0, keepdim=True).values + 1e-8)
X_test_norm = X_test_reduced / (X_test_reduced.max(dim=0, keepdim=True).values + 1e-8)

X_train_norm = X_train_norm.clamp(0, np.pi)
X_test_norm = X_test_norm.clamp(0, np.pi)

print(f"Features d'entraînement : {X_train_norm.shape}")
print(f"Features de test : {X_test_norm.shape}")
print(f"Classes sélectionnées : {selected_classes}")

## Partie B — Tête quantique VQC

In [ ]:
n_layers = 3

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

weight_shapes = {"weights": (n_layers, n_qubits)}
qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes=weight_shapes)

print(f"Paramètres du circuit quantique : {n_layers * n_qubits}")
print(f"Nombre de qubits : {n_qubits}")
print(f"Nombre de couches : {n_layers}")

In [ ]:
class HybridModel(nn.Module):
    def __init__(self, quantum_head, n_qubits, num_classes):
        super().__init__()
        self.quantum_head = quantum_head
        self.classifier = nn.Linear(n_qubits, num_classes)

    def forward(self, x):
        x = self.quantum_head(x)
        x = self.classifier(x)
        return x

model_quantum = HybridModel(qlayer, n_qubits, num_classes)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Modèle hybride créé")
print(f"Paramètres entraînables : {count_parameters(model_quantum)}")

## Partie C — Entraînement

In [ ]:
def train_model(model, X_train, y_train, X_test, y_test, epochs, lr):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    test_accuracies = []

    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test)
            preds = torch.argmax(test_outputs, dim=1)
            acc = accuracy_score(y_test.numpy(), preds.numpy())
            test_accuracies.append(acc)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} — Loss: {loss.item():.4f} — Acc: {acc:.4f}")

    elapsed = time.time() - start_time
    return train_losses, test_accuracies, elapsed

quantum_losses, quantum_accs, quantum_time = train_model(
    model_quantum, X_train_norm, y_train, X_test_norm, y_test, n_epochs, lr
)

print(f"\nTemps d'entraînement (quantique) : {quantum_time:.2f}s")
print(f"Accuracy finale (quantique) : {quantum_accs[-1]:.4f}")

## Partie D — Tête classique (baseline)

In [ ]:
class ClassicalHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model_classical = ClassicalHead(512, n_qubits, num_classes)

print(f"Modèle classique créé")
print(f"Paramètres entraînables : {count_parameters(model_classical)}")

In [ ]:
classical_losses, classical_accs, classical_time = train_model(
    model_classical, X_train, y_train, X_test, y_test, n_epochs, lr
)

print(f"\nTemps d'entraînement (classique) : {classical_time:.2f}s")
print(f"Accuracy finale (classique) : {classical_accs[-1]:.4f}")

## Partie E — Comparaison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(quantum_losses, label="Quantique", color="purple")
axes[0].plot(classical_losses, label="Classique", color="blue")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Courbe de loss")
axes[0].legend()

axes[1].plot(quantum_accs, label="Quantique", color="purple")
axes[1].plot(classical_accs, label="Classique", color="blue")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Courbe d'accuracy")
axes[1].legend()

metrics = ["Accuracy", "Temps (s)", "Paramètres"]
quantum_vals = [quantum_accs[-1], quantum_time, count_parameters(model_quantum)]
classical_vals = [classical_accs[-1], classical_time, count_parameters(model_classical)]

x = np.arange(len(metrics))
width = 0.35
axes[2].bar(x - width/2, quantum_vals, width, label="Quantique", color="purple")
axes[2].bar(x + width/2, classical_vals, width, label="Classique", color="blue")
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics)
axes[2].set_title("Comparaison des métriques")
axes[2].legend()

plt.tight_layout()
plt.show()

print("=" * 50)
print(f"{'Métrique':<20} {'Quantique':<15} {'Classique':<15}")
print("=" * 50)
print(f"{'Accuracy':<20} {quantum_accs[-1]:<15.4f} {classical_accs[-1]:<15.4f}")
print(f"{'Temps (s)':<20} {quantum_time:<15.2f} {classical_time:<15.2f}")
print(f"{'Paramètres':<20} {count_parameters(model_quantum):<15} {count_parameters(model_classical):<15}")
print("=" * 50)

## Exercices
1. Tester avec EfficientNet-B0 comme backbone
2. Augmenter le nombre de qubits à 6 et 8
3. Ajouter du bruit dépolarisant (p=0.01, 0.05, 0.1) et observer la dégradation